# 03 · Partición 70/15/15 sin fuga de productos

Paso 4 de la etapa 3. Los 15 012 outfits vestibles se reparten en train / valid / test (70/15/15, decisión del 01/11/2026) con una condición: **ningún producto puede aparecer en dos particiones**. Si un mismo bolso está en un outfit de train y en otro de test, el modelo "ya lo vio" y la evaluación queda inflada.

**Cómo:** se arma un grafo outfit–producto; los outfits unidos por productos compartidos forman un grupo y se mueven siempre juntos. Se parte de la partición oficial de Han et al. (2017), que ya no comparte productos, y se pasan a valid y test grupos completos de train que no tocan el resto de train (`particion_sin_fuga` en `src/modelo_fly/polyvore.py`).

**Entrada:** `polyvore_items_atributos.csv`. **Salida:** `data/processed/hito3/particion_outfits.csv` (se sube a git: es pequeña y fija la partición para todo el equipo).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.polyvore import componentes, fuga, particion_sin_fuga

PROC = ROOT / "data" / "processed" / "hito3"
SALIDA = PROC / "particion_outfits.csv"
items = pd.read_csv(PROC / "polyvore_items_atributos.csv", dtype=str, keep_default_na=False)
outfits = pd.read_csv(PROC / "polyvore_outfits.csv", dtype=str, keep_default_na=False)
outfits = outfits[outfits["valido"] == "True"]
print("Outfits vestibles:", items["set_id"].nunique(), "| prendas:", len(items), "| productos:", items["tid"].nunique())

Outfits vestibles: 15012 | prendas: 81372 | productos: 65366


## 1. Por qué no sirve una partición al azar

In [2]:
comp = componentes(items)
tam = comp.value_counts()
print(f"Grupos de outfits unidos por productos: {len(tam)}")
print(f"Grupo más grande: {tam.iloc[0]} outfits ({tam.iloc[0] / len(comp):.1%}); siguientes: {tam.iloc[1:5].tolist()}")
print(f"Outfits sin ningún producto compartido: {(tam == 1).sum()}")

rng = np.random.default_rng(2026)
azar = pd.Series(rng.permutation(comp.index.to_numpy()))
n = len(azar)
azar = pd.DataFrame({"set_id": azar, "particion": ["train"] * int(0.7 * n) + ["valid"] * (int(0.85 * n) - int(0.7 * n))
                     + ["test"] * (n - int(0.85 * n))})
en_train = set(items.loc[items["set_id"].isin(azar.loc[azar["particion"] == "train", "set_id"]), "tid"])
otros = items[items["set_id"].isin(azar.loc[azar["particion"] != "train", "set_id"])]
contaminados = otros["tid"].isin(en_train).groupby(otros["set_id"]).any()
print(f"Al azar 70/15/15: {contaminados.mean():.1%} de los outfits de valid/test tienen algún producto visto en train "
      f"({fuga(items, azar)} productos en dos particiones).")

Grupos de outfits unidos por productos: 3796
Grupo más grande: 10625 outfits (70.8%); siguientes: [10, 7, 7, 7]
Outfits sin ningún producto compartido: 3382


Al azar 70/15/15: 70.0% de los outfits de valid/test tienen algún producto visto en train (5199 productos en dos particiones).


## 2. Partición sin fuga

In [3]:
particion = particion_sin_fuga(items, (0.70, 0.15, 0.15), semilla=2026)
assert fuga(items, particion) == 0, "Hay productos en más de una partición"

resumen = particion["particion"].value_counts().reindex(["train", "valid", "test"]).to_frame("outfits")
resumen["%"] = (100 * resumen["outfits"] / resumen["outfits"].sum()).round(1)
display(resumen)
print("Productos en más de una partición:", fuga(items, particion))
print("Origen (partición oficial → nueva):")
display(pd.crosstab(particion["particion_oficial"], particion["particion"]))

,outfits,%
particion,,
train,10625,70.8
valid,2194,14.6
test,2193,14.6


Productos en más de una partición: 0
Origen (partición oficial → nueva):


particion,test,train,valid
particion_oficial,,,
test,1693,0,0
train,500,10625,1376
valid,0,0,818


## 3. ¿Las particiones se parecen?

Se compara el tipo de outfit y el número de prendas. Los grupos que se pueden mover son outfits con productos poco repetidos, así que valid/test pueden diferir algo de train.

In [4]:
o = outfits.merge(particion[["set_id", "particion"]], on="set_id")
o["tipo"] = np.where(o["vestido"].astype(int) > 0, "vestido", "arriba+abajo")
comparacion = pd.crosstab(o["particion"], o["tipo"], normalize="index").mul(100).round(1)
comparacion["prendas_promedio"] = o.groupby("particion")["items_moda"].apply(lambda s: s.astype(int).mean()).round(2)
display(comparacion.reindex(["train", "valid", "test"]))

tipo,arriba+abajo,vestido,prendas_promedio
particion,,,
train,75.9,24.1,5.55
valid,63.4,36.6,5.08
test,63.1,36.9,5.14


## 4. Salida

In [5]:
particion[["set_id", "particion", "particion_oficial", "componente"]].to_csv(SALIDA, index=False, encoding="utf-8")
print("Guardado:", SALIDA.relative_to(ROOT), particion.shape)

Guardado: data\processed\hito3\particion_outfits.csv (15012, 4)
